# Практическая работа 13 (Тест)
## Тема: Подготовка факторов модели

### Настройка окружения

In [1834]:
import pandas as pd
import numpy as np

Дан набор данных [auto.csv](./auto.csv), содержащий информацию о характеристиках подержанных автомобилей. Подробно изучить описание атрибутов можно по [ссылке](https://archive.ics.uci.edu/dataset/10/automobile) в источнике.

In [1835]:
auto = pd.read_csv("auto.csv")
auto.head(1)

,symboling,normalized_losses,make,fuel_type,aspiration,num_doors,body_style,drive_wheels,engine_location,wheel_base,...,engine_size,fuel_system,bore,stroke,compression_ratio,horsepower,peak_rpm,city_mpg,highway_mpg,price
0,3,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,13495.0


### Вопрос 1
Определите наличие пропущенных значений. В качестве ответа укажите кол-во пропусков в столбце **peak_rpm**.

In [1836]:
auto["peak_rpm"].isnull().sum()

np.int64(2)

Выполним обработку пропущенных значений:

1. Удалим столбец **normalized_losses**.

In [1837]:
auto = auto.drop(columns=["normalized_losses"])

2. Заполним пропуски в столбце **num_doors** самым часто встречающимся значением.

In [1838]:
most_common_doors = auto["num_doors"].mode()[0]

auto["num_doors"] = auto["num_doors"].fillna(most_common_doors)

3. Заполним пропуски в столбцах **bore** и **stroke** средним значением, округленным до 2 знаков после запятой.

In [1839]:
for col in ["bore", "stroke"]:
    col_mean = auto[col].mean()
    auto[col] = auto[col].fillna(round(col_mean, 2))

4. Заполним пропуски в столбцах **horsepower**, **peak_rpm** и **price** средним значением, округленным до целого.

In [1840]:
for col in ["horsepower", "peak_rpm", "price"]:
    col_mean = auto[col].mean()
    auto[col] = auto[col].fillna(round(col_mean))

### Вопрос 2
Укажите кол-во категориальных столбцов.

In [1841]:
num_categorical = auto.select_dtypes(include=["object", "category"]).shape[1]

num_categorical

10

### Вопрос 3
В нашем наборе данных есть два столбца данных, значения которых представляют собой слова, используемые для представления чисел:

1. количество цилиндров в двигателе (**num_cylinders**);
2. количество дверей в машине (**num_doors**).

С помощью метода `replace` выполните замену текстовых значений их числовыми эквивалентами.

В качестве ответа укажите кол-во автомобилей с шестью цилиндрами в двигателе.

In [1842]:
num_map = {
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5,
    "six": 6,
    "eight": 8,
    "twelve": 12,
}

pd.set_option("future.no_silent_downcasting", True)

auto[["num_cylinders", "num_doors"]] = (
    auto[["num_cylinders", "num_doors"]].replace(num_map).infer_objects(copy=False)
)

count_six_cylinders = (auto["num_cylinders"] == 6).sum()

count_six_cylinders

np.int64(24)

### Вопрос 4
Для столбца **body_style** выполним кодирование меток (*label encoding*), преобразовав каждое значение в столбце в число.

Для этого:

1. преобразуем столбец в категорию с помощью функции `astype('category')`;

In [1843]:
auto["body_style"] = auto["body_style"].astype("category")

2. создадим новый столбец **body_style_cat**, содержащий закодированные значения столбца **body_style** с помощью метода доступа `cat.codes`.

In [1844]:
auto["body_style_cat"] = auto["body_style"].cat.codes

В качестве ответа укажите какой код был присвоен типу кузова хэтчбек (*hatchback*).

In [1845]:
hatchback_code = auto["body_style"].cat.categories.get_loc("hatchback")

hatchback_code

2

### Вопрос 5
С помощью функции `pd.get_dummies()` выполните унитарное кодирование (*One Hot Encoding*) столбцов **body_style** и **drive_wheels**.

В качестве ответа выберите вектор, кодирующий информацию о типе кузова для 72-й машины: [*body_convertible*, *body_hardtop*, *body_hatchback*, *body_sedan*, *body_wagon*].

In [1846]:
auto = pd.get_dummies(
    auto, columns=["body_style", "drive_wheels"], prefix=["body", "drive"]
)

encoded_vector = auto.iloc[72][
    [
        "body_convertible",
        "body_hardtop",
        "body_hatchback",
        "body_sedan",
        "body_wagon",
    ]
].values

encoded_vector.astype(int)

array([1, 0, 0, 0, 0])

### Вопрос 6
В нашем наборе данных есть столбец с именем **engine_type** (тип двигателя), который содержит несколько разных значений.

Например, нам для анализа важно, оснащен ли двигатель верхним распредвалом (*Overhead Cam, OHC*) или нет. Другими словами, разные версии *OHC* одинаковы для этого анализа. С помощью метода доступа `str` и функции `np.where` создайте новый столбец **OHC_Code**, который указывает, есть ли в автомобиле двигатель *OHC*.

In [1847]:
auto["OHC_Code"] = np.where(auto["engine_type"].str.contains("ohc", case=False), 1, 0)

### Вопрос 7
Выполните целевое кодирование марок автомобилей (**make**). Для этого создайте столбец **make_m_enc**, в котором каждая марка кодируется средним значением целевой переменной - **price**, округленным до целого.

В качестве ответа укажите результат кодирование для марки *renault*.

In [1848]:
make_price_mean = auto.groupby("make")["price"].mean().round()

auto["make_m_enc"] = auto["make"].map(make_price_mean)

auto.loc[auto["make"] == "renault", "make_m_enc"].iloc[0]

np.float64(9595.0)

### Вопрос 8
С помощью библиотеки `sklearn` выполните прямое кодирование (*Label Encoding*) значений столбца **fuel_system**. Результат кодирования сохраните в столбце **fuel_system_code**. В качестве ответа укажите, какой код был присвоен топливной системе *mfi*.

In [1849]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

auto["fuel_system_code"] = le.fit_transform(auto["fuel_system"])

mfi_code = le.transform(["mfi"])[0]

mfi_code

np.int64(4)

### Вопрос 9
Выполните *One-Hot* кодирование данных столбцов **fuel_type**, **aspiration**, **engine_location** с помощью библиотеки `sklearn`. Добавьте закодированные столбцы к исходному датафрейму по шаблону: **fuel_type_category1**, **fuel_type_category2** и т.д. В качестве ответа укажите кол-во новых столбцов.

In [1850]:
from sklearn.preprocessing import OneHotEncoder

cols_to_encode = ["fuel_type", "aspiration", "engine_location"]

ohe = OneHotEncoder(sparse_output=False, dtype=int)

encoded = ohe.fit_transform(auto[cols_to_encode])

encoded_cols = ohe.get_feature_names_out(cols_to_encode)
encoded_df = pd.DataFrame(encoded, columns=encoded_cols)

auto = auto.join(encoded_df)

encoded_df.shape[1]

6

Теперь мы с вами выполнили кодирование всех категориальных факторов. Для продолжения анализа сохраним в новый датафрейм *auto_model* данные всех столбцов, кроме исходных **make**, **fuel_type**, **aspiration**, **engine_location**, **engine_type**, **fuel_system**.

In [1851]:
drop_cols = [
    "make",
    "fuel_type",
    "aspiration",
    "engine_location",
    "engine_type",
    "fuel_system",
]

auto_model = auto.drop(columns=drop_cols)

### Вопрос 10
Выполним стандартизацию всех факторов, на основе которых будем предсказывать значение цены, и сохраним результат в переменную *auto_zscore*:

In [1852]:
from sklearn.preprocessing import StandardScaler

auto_zscore = auto_model.copy()
del auto_zscore["price"]

auto_zscore = pd.DataFrame(
    StandardScaler().fit_transform(auto_zscore), columns=auto_zscore.columns
)

В качестве ответа укажите стандартизированное значение длины для 4-го автомобиля (нумерация начинается с 0), округлённое до 2 знаков после запятой.

In [1853]:
round(auto_zscore.iloc[3]["length"], 2)

np.float64(0.21)

### Вопрос 11
Выполните разбиение набора данных на обучающую и тестовую выборки:

In [1854]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    auto_zscore, auto_model.price, test_size=0.2, random_state=42
)

После этого создайте модель линейной регрессии, где в качестве предикторов выступают все переменные, целевой переменной является цена автомобиля. В качестве ответа укажите коэффициент перед значением параметра **symboling**, округлённый до 2 знаков после запятой.

In [1855]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

symboling_coeff = model.coef_[X_train.columns.get_loc("symboling")]

round(symboling_coeff, 2)

np.float64(213.54)

### Вопрос 12
Оцените качество полученной модели с использованием метрики *MSE*. В качестве ответа укажите получившееся значение, округленное до 2 знаков после запятой.

In [1856]:
from sklearn.metrics import mean_squared_error

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)

round(mse, 2)

13983699.97

### Вопрос 13

1. Выполним отбор факторов, оставив только 8 первых наиболее коррелирующих по абсолютному значению с ценой. 

In [1857]:
corr_matrix = auto_model.corr()

price_corr = corr_matrix["price"].abs()

top_8_corr = price_corr.sort_values(ascending=False).head(9)

top_8_corr = top_8_corr.drop("price")

top_8_columns = top_8_corr.index

2. Выполним построение линейной модели с использованием данных факторов.

In [1858]:
X_train_selected = X_train[top_8_columns]
X_test_selected = X_test[top_8_columns]

model = LinearRegression()
model.fit(X_train_selected, y_train)

LinearRegression()

В качестве ответа укажите значение корреляции между ценой (**price**) и параметром **curb_weight**, округленное до 2 знаков после запятой.

In [1859]:
corr_curb_weight = corr_matrix["price"]["curb_weight"]
round(corr_curb_weight, 2)

np.float64(0.82)

### Вопрос 14
Оцените качество второй модели с использованием метрики *MAE*. В качестве ответа укажите получившееся значение, округленное до 2 знаков после запятой.

In [1860]:
from sklearn.metrics import mean_absolute_error

y_pred = model.predict(X_test_selected)

mae = mean_absolute_error(y_test, y_pred)

round(mae, 2)

2284.05

### Вопрос 15
Выберите выводы, которые можно сделать на основе проведенного анализа.

*Варианты ответа:*

1. унитарное кодирование (*One Hot Encoding*) заключается в том, что каждой категории сопоставляется некоторое числовое значение - код;

2. качество второй модели, основанной только на наиболее коррелирующих факторах с целевой переменной, выше, чем первой;

3. недостатком кодирования меток (*label encoding*) является то, что числовые значения могут быть "неверно интерпретированы" алгоритмами;

4. метод стандартизации масштабирует значения переменных так, чтобы они находились в диапазоне от 0 до 1.

*Ответ:* 2, 3